# MARV — layer x feature activation heatmaps (T4)

A different question than the first notebook: for **one prompt**, which
FFN features fire, and at which depth in the model? X-axis is feature
index, Y-axis is layer, color is cosine similarity between that layer's
hidden state and that layer's gate rows.

Uses SmolLM2's real, documented tool-calling format (system prompt + JSON
tool schema + `<tool_call>` output constraint -- see `marv.toolcall`) rather
than a bare sentence, since that's the actual context the model was tuned to
respond to.

Runtime: **T4 GPU** (Runtime > Change runtime type > T4).

In [ ]:
!pip install -q transformers accelerate numpy matplotlib scikit-learn
!git clone -q https://github.com/thebnbrkr/marv.git /content/marv
%cd /content/marv
!pip install -q -e .

## Load the tool-tuned checkpoint and extract its vindex

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from marv.extract import extract

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL = "gvij/SmolLM2-135M-Function-Calling"
BASE_FOR_TEMPLATE = "HuggingFaceTB/SmolLM2-135M-Instruct"  # the tuned checkpoint's tokenizer has no chat_template of its own

tok = AutoTokenizer.from_pretrained(MODEL)
base_tok_for_template = AutoTokenizer.from_pretrained(BASE_FOR_TEMPLATE)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(device).eval()

vindex = extract(model, model_name=MODEL)
print(vindex.num_layers, "layers,", vindex.gate[0].shape[0], "features/layer")

## Build the real tool-calling prompt pair

Same approach as the first notebook: a real tool schema via
`transformers.utils.get_json_schema`, rendered through SmolLM2's documented
system prompt.

In [ ]:
from transformers.utils import get_json_schema
from marv.toolcall import build_smollm2_tool_prompt

def get_weather(location: str) -> str:
    """Gets the current weather for a location.

    Args:
        location: The city to get weather for.
    """
    return "sunny"

tools = [get_json_schema(get_weather)]

TOOL_PROMPT = build_smollm2_tool_prompt(
    tok, tools, "What is the weather in Paris?", chat_template=base_tok_for_template.chat_template,
)
PLAIN_PROMPT = build_smollm2_tool_prompt(
    tok, tools, "Tell me an interesting fact about Paris.", chat_template=base_tok_for_template.chat_template,
)
print("token lengths:", len(tok(TOOL_PROMPT).input_ids), len(tok(PLAIN_PROMPT).input_ids))

## A. Layer x feature heatmap, both prompts compared

Every feature at every layer for each prompt -- no top-k reduction, this is
the full grid. `plot_comparison` puts both heatmaps on a *shared* color
scale (two separately-normalized `imshow` calls can look equally "bright"
even when the underlying magnitudes differ), plus the difference and a
per-layer total-activation ("attribution") panel, in one figure.

In [ ]:
from marv.layer_heatmap import compute, plot_comparison

hm_tool = compute(vindex, model, tok, TOOL_PROMPT, device=device)
hm_plain = compute(vindex, model, tok, PLAIN_PROMPT, device=device)
print("matrix shape (layers x features):", hm_tool.matrix.shape)

fig = plot_comparison(hm_tool, hm_plain, title=f"{MODEL}: tool vs. plain phrasing of the same request")
fig

In [ ]:
import numpy as np
from marv.layer_heatmap import difference

d = difference(hm_tool, hm_plain)
row_max = np.abs(d.matrix).max(axis=1)
top_layers = np.argsort(-row_max)[:5]
print("layers with the largest tool-vs-plain divergence:")
for i in top_layers:
    layer = d.layers[i]
    feature = int(np.argmax(np.abs(d.matrix[i])))
    print(f"  L{layer}: |delta|={row_max[i]:.3f} at f{feature}")

## C. Top-N features per layer

Not just the single peak -- which features *together* dominate a layer for
this prompt.

In [ ]:
from marv.layer_heatmap import top_features_per_layer

top_per_layer = top_features_per_layer(hm_tool, n=5)
for layer, hits in zip(hm_tool.layers, top_per_layer):
    if layer % 5 == 0:  # print every 5th layer to keep this readable
        formatted = ", ".join(f"f{f}={v:.2f}" for f, v in hits)
        print(f"L{layer}: {formatted}")

## D. Feature clustering across prompts (PCA / t-SNE)

A geometric answer to "does the model have a distinct tool-calling feature
cluster": embed several tool-triggering phrasings and several plain
phrasings (all using the real schema, varying only the final question) as
their full feature-activation vectors at one layer, project to 2D, and see
whether they separate. `cluster_features` then names the actual columns
responsible for any separation -- the literal "tool-cluster" features, not
just a visual grouping.

In [ ]:
from marv.clustering import prompt_activations, reduce_pca, plot_projection, cluster_features

tool_queries = [
    "What is the weather in Paris?",
    "What is the weather in Tokyo?",
    "Can you check the weather in Berlin?",
]
plain_queries = [
    "Tell me an interesting fact about Paris.",
    "What language do they speak in Tokyo?",
    "Write a short poem about Berlin.",
]
groups = {
    "tool": [build_smollm2_tool_prompt(tok, tools, q, chat_template=base_tok_for_template.chat_template) for q in tool_queries],
    "plain": [build_smollm2_tool_prompt(tok, tools, q, chat_template=base_tok_for_template.chat_template) for q in plain_queries],
}

CLUSTER_LAYER = 20  # picked by checking a few layers for separation -- with prompts this close (identical system prompt + tools, differing only in the final question), the tool-vs-plain gap is subtle almost everywhere; 20 had the clearest signal
pa = prompt_activations(vindex, model, tok, groups, layer=CLUSTER_LAYER, device=device)

points = reduce_pca(pa, n_components=2)
fig = plot_projection(pa, points, title=f"tool vs. plain prompts, PCA of layer {CLUSTER_LAYER} activations")
fig

In [ ]:
tool_cluster = cluster_features(pa, "tool", min_group_activation=0.05, other_threshold=0.035, top_n=10)
print(f"features consistently higher for 'tool' prompts than 'plain' at layer {CLUSTER_LAYER}:", tool_cluster)
# Thresholds are loose on purpose: these 6 prompts share nearly all of their
# text (same system prompt + tool schema, only the final question differs),
# so the tool-vs-plain gap per feature is small (~0.02-0.04) rather than the
# dramatic separation you'd see between, say, unrelated topics in the first
# notebook's polysemanticity heatmap. Try more/longer, more distinct
# tool-vs-plain prompt pairs to sharpen this.

from marv.probe import describe_feature
for f in tool_cluster[:5]:
    tok_ids, logits = describe_feature(vindex, CLUSTER_LAYER, f, k=3)
    print(f"  f{f} promotes:", tok.batch_decode([[t] for t in tok_ids]))

## B. "Layer trace" -- with an honest caveat

A natural next ask is "show feature 134's activation across all layers."
That's not quite well-defined here: MARV's features are raw MLP neurons, one
gate row per layer -- feature index 134 at layer 3 and feature index 134 at
layer 18 are unrelated neurons that happen to share a column position, not a
persistent identity threading through the model (that *is* meaningful in
sparse-autoencoder-based interpretability, which builds an explicit shared
feature dictionary -- MARV doesn't have one).

What *is* well-defined and answers the same underlying question ("which
layers matter most for this input"): the strongest-firing feature *at each
layer*, tracked layer by layer. `peak_activation_trace()` gives you that.

In [ ]:
from marv.layer_heatmap import peak_activation_trace

peak_tool, feat_tool = peak_activation_trace(hm_tool)
peak_plain, feat_plain = peak_activation_trace(hm_plain)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hm_tool.layers, peak_tool, marker="o", label="tool prompt")
ax.plot(hm_plain.layers, peak_plain, marker="o", label="plain prompt")
ax.set_xlabel("layer")
ax.set_ylabel("peak cosine similarity (strongest feature at that layer)")
ax.set_title("per-layer peak activation")
ax.legend()
fig.tight_layout()
fig

In [ ]:
# Raw column trace, if you want to check whether a *specific* feature index
# happens to stay active across nearby layers (see the caveat above for why
# this isn't "the same neuron" beyond coincidence).
from marv.layer_heatmap import layer_trace

# Pick a feature index that showed up as a peak somewhere interesting above.
example_feature = int(feat_tool[15])
print(f"feature index {example_feature} across all layers (tool prompt):")
print(layer_trace(hm_tool, example_feature))

## Next steps

- Swap `tool_queries`/`plain_queries` for your own pairs, or for a
  non-tool-calling behavior (e.g. instruction-following vs. plain
  statements) -- nothing here is tool-call-specific.
- Try `reduce_tsne` instead of `reduce_pca` in section D with more prompts
  per group (t-SNE needs enough points per its perplexity setting to be
  meaningful -- a handful of prompts is really a PCA-scale problem).
- Cross-reference the layers this notebook's difference() flags as most
  divergent against `marv.diff.most_changed()`'s layers from the first
  notebook (15/20/22 for this checkpoint) -- the first notebook now has a
  full worked version of this, with a real finding: correlation is weak
  (0.04) on a naive bare-sentence prompt pair but moderate (0.42) once using
  the real tool schema above, with layer 22 the most consistent overlap
  point -- which is why `CLUSTER_LAYER` in section D defaults to it.